In [1]:
import pandas as pd
from config.conexiones import get_consulta, get_engine
from datetime import datetime, timedelta, timezone
pd.set_option('display.max_rows',None) 
pd.set_option('display.max_columns', None) 
from unidecode import unidecode

ww = 'C:/Users/Usuario/Downloads/'  # carpeta principal  con los archivos


## Blue Express


In [326]:
be2 = pd.read_excel(ww + "Matriz de Transito Bluexpress al 31-03-2026.xlsx", header = 2) 


In [328]:
be2[be2['COD COMUNA'] == 5505]

,Unnamed: 0,COD REGION,NOM REGION,COD COMUNA,NOM COMUNA,POSTA COMUNA,EXPRESS,PRIORITY,*Tiempos de promesa sujetos a revisión según realidad de los tiempos de despachos.
59,NaN,5,Valparaiso,5505,LIMACHE,LIC,6,5,NaN


In [378]:
be = pd.read_excel(ww + "Matriz de Transito Bluexpress al 31-03-2026.xlsx", header = 2) 


In [379]:
be['geo_origen_id'] = 333  
be['geo3_codigo'] = be['COD COMUNA'].apply(lambda x: str(x) if len(str(x)) == 5 else '0' + str(x))
be['mensaje'] = "Tiempos de promesa sujetos a revisión según realidad de los tiempos de despachos."
cols = ['geo_origen_id', 'geo3_codigo', 'COD COMUNA', 'NOM COMUNA',  'EXPRESS', 'PRIORITY' , 'mensaje' ]
be = be[cols]

In [380]:
be = pd.melt(be, id_vars=['geo_origen_id', 'geo3_codigo', 'COD COMUNA', 'NOM COMUNA', 'mensaje'], value_vars=['EXPRESS', 'PRIORITY'], var_name= 'Servicio', value_name= 'nro_dias')
be['servicio_transporte_id'] = be.Servicio.apply(lambda x: 1 if x == 'EXPRESS' else 2)


In [381]:
rel_comunas = get_consulta('dwh', "select id::integer as geo4_destino_id, split_part(codigo, ' - ', 1) as geo3_codigo, nombre as geo3_nombre " \
"from dwh.dim_geografia_nivel4 " \
"where split_part(codigo, ' - ', 2) = '000'  " \
"")
rel_comunas.sample(3)

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:61: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,geo4_destino_id,geo3_codigo,geo3_nombre
308,309,13101,Santiago Centro
276,277,07306,Romeral
246,247,10207,Queilén


In [382]:
# be['geo_destino_id'].value_counts(dropna = False)
# be[be['geo_destino_id'].isna()]
rel_comunas.shape

(346, 3)

In [383]:
be = be.merge(rel_comunas, how = 'left', on = 'geo3_codigo')
# be['geo_destino_id'] = be['geo_destino_id'].astype(int) 
be.isna().sum()

geo_origen_id             0
geo3_codigo               0
COD COMUNA                0
NOM COMUNA                0
mensaje                   0
Servicio                  0
nro_dias                  0
servicio_transporte_id    0
geo4_destino_id           8
geo3_nombre               8
dtype: int64

In [248]:

# be[be['NOM COMUNA'].str.lower() != be.geo3_nombre.str.lower()][['NOM COMUNA', 'geo3_nombre']].drop_duplicates()


In [384]:
be[be.geo4_destino_id.isna()]
d = {'05505' : 140, 
     '05507' : 191, 
     '05106': 255, 
     '05108' : 339
       }

be['geo4_destino_id'] = be.apply(lambda x: d[x['geo3_codigo']] if pd.isna(x['geo4_destino_id']) else x['geo4_destino_id'] ,axis = 1)

In [385]:
be[be.geo4_destino_id.isna()]

,geo_origen_id,geo3_codigo,COD COMUNA,NOM COMUNA,mensaje,Servicio,nro_dias,servicio_transporte_id,geo4_destino_id,geo3_nombre


In [386]:

drop_cols = ['geo3_codigo', 'COD COMUNA',	'NOM COMUNA', 'mensaje', 'Servicio', 'geo3_nombre']
be.drop(drop_cols, axis = 1, inplace = True)

In [387]:
be = be.fillna(333)
be['geo4_destino_id'] = be['geo4_destino_id'].astype(int)

In [388]:
be.sample()
# be.isna().sum()

,geo_origen_id,nro_dias,servicio_transporte_id,geo4_destino_id
33,333,6,1,112


In [389]:
be['fecha_desde'] = datetime.now(timezone.utc) - timedelta(days = 10*365)

In [390]:
be.sample()

,geo_origen_id,nro_dias,servicio_transporte_id,geo4_destino_id,fecha_desde
430,333,3,2,24,2016-04-17 15:10:51.041016+00:00


In [391]:
be.columns = ['geo4_origen_id', 'nro_dias', 'servicio_transporte_id', 'geo4_destino_id', 'fecha_desde']

In [392]:
be.shape

(692, 5)

In [393]:
be.nro_dias.value_counts()

nro_dias
6     241
5     198
7      87
3      51
4      46
8      27
9      10
17      7
20      6
2       5
15      5
10      3
11      3
16      2
18      1
Name: count, dtype: int64

In [367]:
# be.geo4_destino_id.value_counts()

In [363]:
892/2

446.0

In [394]:
be.to_sql('dim2_promesas_servicios_transportistas', get_engine('dwh'), index = False, if_exists='append', schema = 'dwh')

692

## Correos de Chile


In [510]:
cch = pd.read_excel(ww + "Matriz de Transito Correos de Chile..xlsx", header = 1).drop_duplicates()
cch.head(2)
# cch['Planta Destino'].value_counts()

,Zona Origen,Planta Origen,Adm. Oficina Origen,Zona Destino,Planta Destino,Nombre Comuna Destino,Nombre Comuna/Localidad de Destino,Tipo,Distribución Domiciliaria,Distribución Suc / Age,Canal Cartero,Canal Móvil,Sucursal,Agencia,Restricción de Volumen y peso
0,CENTRO,VIÑA DEL MAR,SUC,CENTRO,PLANTA CEP RM,ALGARROBO,ALGARROBO,COMUNA,1,1,SI,SI,SI,NO,NaN
1,CENTRO,VIÑA DEL MAR,SUC,CENTRO,PLANTA CEP RM,ALHUE,ALHUE,COMUNA,1,1,NO,SI,NO,SI,NaN


In [511]:
cch['geo4_origen_id'] = 341

In [512]:
cols = ['geo4_origen_id', 'Planta Destino',  'Nombre Comuna Destino', 'Nombre Comuna/Localidad de Destino',	'Tipo', 
        'Distribución Domiciliaria', 'Distribución Suc / Age']
cch = cch[cols] 
cch.columns = ['geo4_origen_id', 'planta', 'comuna', 'localidad', 'tipo', 'Domiciliario', 'Sucursal/Agencia']

In [513]:
cch2 = cch.copy() 
cch.sample(3)

,geo4_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia
177,341,LA CALERA,LOS VILOS,QUILIMARI,LOCALIDAD,Sin Distribución,1
17,341,PLANTA CEP RM,BUIN,ALTO JAHUEL,LOCALIDAD,1,Sin distribución
94,341,RANCAGUA,DOÑIHUE,LO MIRANDA,LOCALIDAD,Sin Distribución,1


In [514]:
cch2.tipo.value_counts()

tipo
COMUNA       320
LOCALIDAD     75
Name: count, dtype: int64

In [14]:
rel_comunas = get_consulta('dwh', "select id::integer as geo4_destino_id, split_part(codigo, ' - ', 1) as geo3_codigo, nombre as geo3_nombre " \
"from dwh.dim_geografia_nivel4 " \
"where split_part(codigo, ' - ', 2) = '000'  " \
"")

rel_localidades = get_consulta('dwh', "select id::integer as geo4_destino_id, split_part(codigo, ' - ', 1) as geo3_codigo, upper(nombre) as geo4_nombre " \
"from dwh.dim_geografia_nivel4 " \
"--where split_part(codigo, ' - ', 2) = '000'  " \
"")
rel_localidades['geo4_nombre'] = rel_localidades.geo4_nombre.apply(lambda x: unidecode(x))


rel_plantas = get_consulta('dwh', "select id::integer as geo4_sucursal_destino_id, upper(nombre) as geo4_nombre_sucursal from dwh.dim_geografia_nivel4")
rel_plantas['geo4_nombre_sucursal'] = rel_plantas.geo4_nombre_sucursal.apply(lambda x: unidecode(x))

rel_comunas.sample(3)
rel_localidades.sample(3)

rel_plantas.sample()

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,geo4_sucursal_destino_id,geo4_nombre_sucursal
46,48,CHONCHI


In [516]:
cch.sample()

,geo4_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia
69,341,PLANTA CEP RM,CONCHALI,CONCHALI,COMUNA,1,1


In [517]:
rel_localidades.shape

(449, 3)

In [518]:
rel_localidades.iloc[415:420]

,geo4_destino_id,geo3_codigo,geo4_nombre
415,419,13401,NOS
416,421,11202,PUERTO RAUL MARIN BALMACEDA
417,422,03301,DOMEYCO
418,423,10101,ALERCE
419,424,05502,ARTIFICIO


In [519]:
test = cch.merge(rel_localidades, how = 'left', left_on = 'localidad', right_on = 'geo4_nombre')

In [520]:
cch.shape

(395, 7)

In [521]:
# ESTACION PAIPOTE               2
# EL QUISCO 

In [522]:
test.shape

(395, 10)

In [523]:
test.isna().sum()

geo4_origen_id      0
planta              0
comuna              0
localidad           0
tipo                0
Domiciliario        0
Sucursal/Agencia    0
geo4_destino_id     8
geo3_codigo         8
geo4_nombre         8
dtype: int64

In [477]:
np.sort(test[test.geo4_destino_id.isna()].localidad.unique())

array(['CAPITAN PASTEN', 'ESTACION DOMEIKO', 'ESTACION PAIPOTE',
       'SAN GREGORIO (XVI)'], dtype=object)

In [524]:
d = {'COYHAIQUE' : 55, 
     'LA CALERA' : 23, 
     'LLAY LLAY' : 143, 
     'PAIHUANO' : 197, 
     'SANTIAGO' : 309, 
     'VIÑA DEL MAR' : 341, 
     'CAPITAN PASTEN' : 417,
     'ESTACION DOMEIKO' : 422 , 
    #  'ESTACION PAIPOTE' : 407,  #452
     'SAN GREGORIO (XVI)' : 415
    }

test['geo4_destino_id'] = test.apply(lambda x: d[x['localidad']] if x['localidad'] in d.keys() else x['geo4_destino_id'] ,axis = 1)



In [525]:
test.isna().sum()

geo4_origen_id      0
planta              0
comuna              0
localidad           0
tipo                0
Domiciliario        0
Sucursal/Agencia    0
geo4_destino_id     0
geo3_codigo         8
geo4_nombre         8
dtype: int64

In [526]:
rel_plantas.sample()

,geo4_sucursal_destino_id,geo4_nombre_sucursal
395,399,CUMPEO


In [527]:
test['planta'] = test.planta.replace('PLANTA CEP RM', 'RENCA')
test = test.merge(rel_plantas, how = 'left', left_on = 'planta', right_on = 'geo4_nombre_sucursal')

In [528]:

test['geo4_sucursal_destino_id'] = test.apply(lambda x: d[x['planta']] if x['planta'] in d.keys() else x['geo4_sucursal_destino_id'] ,axis = 1)
test.sample(2)

,geo4_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia,geo4_destino_id,geo3_codigo,geo4_nombre,geo4_sucursal_destino_id,geo4_nombre_sucursal
257,341,VIÑA DEL MAR,PUCHUNCAVI,MAITENCILLO,LOCALIDAD,1,Sin distribución,404.0,05105,MAITENCILLO,341.0,NaN
142,341,RENCA,LA PINTANA,LA PINTANA,COMUNA,1,1,125.0,13112,LA PINTANA,266.0,RENCA


In [491]:
# test[test['geo4_nombre_sucursal'].isna()].planta.unique()

In [529]:
test.isna().sum()

geo4_origen_id               0
planta                       0
comuna                       0
localidad                    0
tipo                         0
Domiciliario                 0
Sucursal/Agencia             0
geo4_destino_id              0
geo3_codigo                  8
geo4_nombre                  8
geo4_sucursal_destino_id     0
geo4_nombre_sucursal        58
dtype: int64

### Comunas

In [389]:
cch = cch2[cch2.tipo == 'COMUNA']
cch.drop(['localidad', 'tipo'], axis = 1, inplace = True)





C:\Users\Usuario\AppData\Local\Temp\ipykernel_24316\1831072211.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cch.drop(['localidad', 'tipo'], axis = 1, inplace = True)


In [390]:
cch.shape


(320, 5)

In [391]:
rel_comunas = get_consulta('dwh', "select id::integer as geo_destino_id,  geo3_nombre from dwh.dim_geografia")
rel_plantas = get_consulta('dwh', "select id::integer as geo_sucursal_id, geo3_nombre as geo3_nombre_sucursal from dwh.dim_geografia")
rel_comunas.sample(3)
rel_plantas.sample()

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:61: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,geo_sucursal_id,geo3_nombre_sucursal
123,124,La Ligua


In [392]:
rel_comunas['geo3_nombre'] = rel_comunas.geo3_nombre.apply(lambda x: unidecode(x).upper())
rel_plantas['geo3_nombre_sucursal'] = rel_plantas.geo3_nombre_sucursal.apply(lambda x: unidecode(x).upper())
rel_plantas.sample(2)

,geo_sucursal_id,geo3_nombre_sucursal
345,346,ZAPALLAR
175,176,MONTE PATRIA


In [393]:
cch = cch.merge(rel_comunas, how = 'left', left_on = 'comuna', right_on = 'geo3_nombre')

In [394]:
d = {'COYHAIQUE' : 55, 
     'LA CALERA' : 23, 
     'LLAY LLAY' : 143, 
     'PAIHUANO' : 197, 
     'SANTIAGO' : 309, 
     'VIÑA DEL MAR' : 341}

cch['geo_destino_id'] = cch.apply(lambda x: d[x['comuna']] if x['comuna'] in d.keys() else x['geo_destino_id'] ,axis = 1)


In [395]:
cch.shape

(320, 7)

In [396]:
cch[cch.geo_destino_id.isna()]

,geo_origen_id,planta,comuna,Domiciliario,Sucursal/Agencia,geo_destino_id,geo3_nombre


In [397]:
cch['planta'] = cch.planta.replace('PLANTA CEP RM', 'RENCA')
cch = cch.merge(rel_plantas, how = 'left', left_on = 'planta', right_on = 'geo3_nombre_sucursal')

In [398]:

cch['geo_sucursal_id'] = cch.apply(lambda x: d[x['planta']] if x['planta'] in d.keys() else x['geo_sucursal_id'] ,axis = 1)
cch.sample(2)

,geo_origen_id,planta,comuna,Domiciliario,Sucursal/Agencia,geo_destino_id,geo3_nombre,geo_sucursal_id,geo3_nombre_sucursal
105,341,RENCA,ISLA DE PASCUA,Sin Distribución,8,116.0,ISLA DE PASCUA,266.0,RENCA
26,341,CASTRO,CASTRO,2,2,33.0,CASTRO,33.0,CASTRO


In [399]:
cch.sample(20)
cch.isna().sum()
# cch[cch.geo_sucursal_id.isna()]

geo_origen_id            0
planta                   0
comuna                   0
Domiciliario             0
Sucursal/Agencia         0
geo_destino_id           0
geo3_nombre              5
geo_sucursal_id          0
geo3_nombre_sucursal    44
dtype: int64

In [530]:
cch = test.copy()

In [531]:
cols = ['geo4_origen_id',  'Domiciliario', 'Sucursal/Agencia',  'geo4_destino_id',  'geo4_sucursal_destino_id']
cch = cch[cols] 
cch = pd.melt(cch, id_vars=['geo4_origen_id',  'geo4_destino_id',  'geo4_sucursal_destino_id'], value_vars=['Domiciliario', 'Sucursal/Agencia'], value_name='nro_dias', var_name='Servicio')

cch.sample(2)
cch.columns 

Index(['geo4_origen_id', 'geo4_destino_id', 'geo4_sucursal_destino_id',
       'Servicio', 'nro_dias'],
      dtype='object')

In [532]:
cch['servicio_transporte_id'] = cch.Servicio.apply(lambda x: 3 if x == 'Domiciliario' else 4  )
cch['geo4_destino_id']  = cch['geo4_destino_id'].astype(int)
cch['geo4_sucursal_destino_id'] = cch['geo4_sucursal_destino_id'].astype(int)
cch.drop(['Servicio'], axis = 1, inplace = True)
cch.sample(2)

,geo4_origen_id,geo4_destino_id,geo4_sucursal_destino_id,nro_dias,servicio_transporte_id
657,341,235,235,1,4
465,341,66,341,1,4


In [533]:
cch.rename(columns  = {'geo_sucursal_id' : 'geo_sucursal_destino_id' } , inplace=True)

In [534]:
cch['fecha_desde'] = datetime.now(timezone.utc) - timedelta(days = 365*10)

In [535]:
cch = cch[~cch.nro_dias.isin(['Sin Distribución', 'Sin distribución'])]

In [536]:
cch.nro_dias.unique()

array([1, 2, 3, 4, 7, 5, 6, 8, 16], dtype=object)

In [537]:
cch[cch.geo4_destino_id == 407]
test[test.geo4_destino_id == 407]

,geo4_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia,geo4_destino_id,geo3_codigo,geo4_nombre,geo4_sucursal_destino_id,geo4_nombre_sucursal
75,341,COPIAPO,COPIAPO,PAIPOTE,LOCALIDAD,1,Sin distribución,407.0,03101,PAIPOTE,69.0,COPIAPO


In [538]:
# cch.sample()
cch.to_sql('dim2_promesas_servicios_transportistas', get_engine('dwh'), index = False, if_exists='append', schema = 'dwh')

614

### Localidades


In [81]:
cch = cch2[cch2.tipo != 'COMUNA']

In [82]:
cch.head()

,geo_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia
5,341,PUERTO MONTT,ANCUD,CHACAO,LOCALIDAD,Sin Distribución,1
11,341,CONCEPCION,ARAUCO,CARAMPANGUE,LOCALIDAD,Sin Distribución,1
12,341,CONCEPCION,ARAUCO,LARAQUETE,LOCALIDAD,Sin Distribución,1
15,341,COYHAIQUE,AYSEN,PUERTO AGUIRRE,LOCALIDAD,Sin Distribución,7
16,341,COYHAIQUE,AYSEN,PUERTO CHACABUCO,LOCALIDAD,3,Sin distribución


In [83]:
cch.localidad.unique()

array(['CHACAO', 'CARAMPANGUE', 'LARAQUETE', 'PUERTO AGUIRRE',
       'PUERTO CHACABUCO', 'ALTO JAHUEL', 'EL RECURSO',
       'VALDIVIA DE PAINE', 'VILUCO', 'PUERTO WILLIAMS', 'MONTE AGUILA',
       'TROVOLHUE', 'SAN SEBASTIAN', 'LA JUNTA', 'CASAS DE CHACABUCO',
       'ESTACION PAIPOTE', 'PAIPOTE', 'TONGOY', 'LOS LAURELES',
       'EL SALVADOR', 'LO MIRANDA', 'EL PAICO', 'LAS CRUCES', 'QUEPE',
       'HUALPIN', 'MELINKA', 'HORNOPIREN', 'LONQUEN', 'SANTA FE DE LAJA',
       'BATUCO', 'MALALHUE', 'QUILIMARI', 'CAPITAN PASTEN', 'PELEQUEN',
       'BOLLENAR', 'MALLARAUCO', 'SAN JOSE DE MELIPILLA', 'EL PALQUI',
       'SAN GREGORIO (XVI)', 'MALLOCO', 'MONTEGRANDE', 'PISCO ELQUI',
       'HOSPITAL', 'CURANIPE', 'MAITENCILLO', 'ENTRE LAGOS',
       'SAN PEDRO DE QUILLOTA', 'EL BELLOTO', 'ACHAO', 'NIPAS', 'ROSARIO',
       'CUMPEO', 'PUERTO RIO TRANQUILO', 'PUERTO INGENIERO IBANEZ',
       'LLO LLEO', 'NOS', 'ISLA NEGRA', 'LABRANZA', 'BARROS ARANA',
       'HUERTO FAMILIARES', 'MONTENEGRO', '

## Chile Express 


### Carga final

In [ ]:

rel4.shape

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


(845, 7)

In [67]:
aux3.sample()
aux3['TIEMPO ENTREGA'].unique()

array(['PRIORITARIO', 'EXPRESS', 'EXTREMOS', 'Ampliado', 'EXTENDIDO',
       'Ampliado Plus', 'Extremo Sur'], dtype=object)

In [ ]:
rel4 = get_consulta('dwh', "select valor_externo, geografia_id as geo4_destino_id from dwh.relacion_geografia where origen = 'CHX-EPE' and nivel_interno = 4 ")
s = {'PRIORITARIO' : 5, 
     'EXPRESS' : 6,
    'EXTREMOS' : 7, 
    'Ampliado' : 8,
    'EXTENDIDO' : 9,
    'Ampliado Plus' : 10, 
    'Extremo Sur' : 11
}

BIGT3
CHEX
XTEN
XTRE


chx = pd.read_excel(ww + "Tiempos Entrega CHX 2026.xlsx") 
chx['servicio_transporte_id'] = chx['TIEMPO ENTREGA'].apply(lambda x: s[x] if x in s.keys() else x)
chx['geo4_origen_id'] = 333
chx['nro_dias'] = chx.DIAS
chx = chx.merge(rel4, how = 'left', left_on = 'LOCALIDAD DESTINO', right_on = 'valor_externo')

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


In [51]:
chx.isna().sum()

LOCALIDAD ORIGEN             0
LOCALIDAD DESTINO            0
PRODUCTO                     0
TIEMPO ENTREGA               0
DIAS                         0
Unnamed: 5                1040
SERVICIO                  1033
DIAS.1                    1033
servicio_transporte_id       0
geo4_origen_id               0
nro_dias                     0
valor_externo                0
geo4_destino_id              0
dtype: int64

In [60]:
cols = ['servicio_transporte_id', 'geo4_origen_id', 'nro_dias', 'geo4_destino_id']
chx = chx[cols]
chx['fecha_desde'] = datetime.now(timezone.utc) - timedelta(days = 365*10)
chx.sample()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_22172\2152518921.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chx['fecha_desde'] = datetime.now(timezone.utc) - timedelta(days = 365*10)


,servicio_transporte_id,geo4_origen_id,nro_dias,geo4_destino_id,fecha_desde
583,6,333,1,752,2016-04-30 15:36:24.286341+00:00


In [ ]:
chx['']

In [61]:
chx.to_sql('dim2_promesas_servicios_transportistas', get_engine('dwh'), index = False, if_exists = 'append', schema = 'dwh')

40

In [62]:
aux3[aux3['LOCALIDAD DESTINO'].str.upper().str.contains('TOCOPI')]

,LOCALIDAD ORIGEN,LOCALIDAD DESTINO,PRODUCTO,TIEMPO ENTREGA,DIAS,Unnamed: 5,SERVICIO,DIAS.1
39,VALPARAISO,TOCOPILLA,ENCOMIENDA,EXTREMOS,4,NaN,NaN,NaN
41,VALPARAISO,TOCOPILLA - CALETA BOY,ENCOMIENDA,EXTREMOS,4,NaN,NaN,NaN


In [37]:
chx.sample()
rel4.sample()
# rel4.valor_externo.value_counts()

,origen,valor_externo,nivel_interno,geografia_id,observaciones,fecha_creacion,fecha_actualizacion
376,CHX-EPE,LOS ANGELES - SANTA EMILIA,4,834.0,Informacion obtenida desde Excel con promesas ...,2026-04-22 13:12:03.449541+00:00,2026-04-22 13:12:03.449541+00:00


In [3]:
rel_localidades = get_consulta('dwh', """
                                select 
                                    g4.id::integer as geo4_destino_id, 
                                    g4.codigo as geo4_codigo, 
                                    --split_part(g4.codigo, ' - ', 1) as geo3_codigo, 
                                    upper(g4.nombre) as geo4_nombre, 
                                    g3.nombre as geo3_nombre
                                from 
                                    dwh.dim_geografia_nivel4 g4 left join 
                                    dwh.dim_geografia_nivel3 g3 on g3.id = g4.nivel3_id
                               """)

rel_localidades['geo4_nombre'] = rel_localidades.geo4_nombre.apply(lambda x: unidecode(x).upper())
rel_localidades['geo3_nombre'] = rel_localidades.geo3_nombre.apply(lambda x: unidecode(x).upper())
rel_localidades['comuna_localidad'] = rel_localidades.apply(lambda x: x['geo4_nombre'] if ' - 000' in x['geo4_codigo'] else f"{x['geo3_nombre']} - {x['geo4_nombre']}" , axis = 1)
rel_localidades.drop(['geo3_nombre', 'geo4_codigo'], axis = 1, inplace = True)
rel_localidades.sample()

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,geo4_destino_id,geo4_nombre,comuna_localidad
310,312,TALAGANTE,TALAGANTE


In [2]:
s = {'PRIORITARIO' : 5, 
     'EXPRESS' : 6,
    'EXTREMOS' : 7, 
    'Ampliado' : 8,
    'EXTENDIDO' : 9,
    'Ampliado Plus' : 10, 
    'Extremo Sur' : 11
}

In [4]:
aux3 = pd.read_excel(ww + "Tiempos Entrega CHX 2026.xlsx")

In [5]:
chx = pd.read_excel(ww + "Tiempos Entrega CHX 2026.xlsx")
chx['valor_externo'] = chx['LOCALIDAD DESTINO']
chx['servicio_transporte_id'] = chx['TIEMPO ENTREGA'].apply(lambda x: s[x] if x in s.keys() else x)
chx['geo4_origen_id'] = 333
chx['nro_dias'] = chx.DIAS
# chx['geo3_nombre'] = chx['LOCALIDAD DESTINO'].apply(lambda x: x.split(' - ')[0])
# chx['localidad'] = chx['LOCALIDAD DESTINO'].apply(lambda x: x.split(' - ')[1] if len(x.split(' - ')) > 1 else x.split(' - ')[0])
# chx['localidad'] = chx.localidad.apply(lambda x: unidecode(x).upper())
chx['LOCALIDAD DESTINO'] = chx['LOCALIDAD DESTINO'].apply(lambda x: unidecode(x).upper())
chx.drop(['LOCALIDAD ORIGEN', 'Unnamed: 5', 'SERVICIO', 'DIAS.1', 'TIEMPO ENTREGA', 'PRODUCTO', 'DIAS'], axis = 1, inplace = True)
chx.sample(10)

,LOCALIDAD DESTINO,valor_externo,servicio_transporte_id,geo4_origen_id,nro_dias
1029,PLACILLA QUINTA REGION,PLACILLA QUINTA REGION,6,333,1
263,CALBUCO,CALBUCO,9,333,3
950,RENACA,RENACA,5,333,1
361,QUINTA DE TILCOCO,QUINTA DE TILCOCO,6,333,1
237,RAUCO,RAUCO,10,333,10
86,CAUQUENES,CAUQUENES,5,333,1
678,LA UNION,LA UNION,6,333,1
481,MELIPILLA - HUECHUN,MELIPILLA - HUECHUN,10,333,10
1036,SAN ANTONIO,SAN ANTONIO,6,333,1
514,SAN VICENTE - LOS MAITENES,SAN VICENTE - LOS MAITENES,10,333,10


In [8]:
# chx2 = chx.copy()
chx = chx2.copy()

In [9]:
chx.sample()

,LOCALIDAD DESTINO,valor_externo,servicio_transporte_id,geo4_origen_id,nro_dias
205,CASTRO - PUTEMUN,CASTRO - PUTEMUN,8,333,7


In [10]:
chx = chx.merge(rel_localidades, how = 'left', left_on = 'LOCALIDAD DESTINO', right_on = 'comuna_localidad')

In [11]:
chx.isna().sum()

LOCALIDAD DESTINO           0
valor_externo               0
servicio_transporte_id      0
geo4_origen_id              0
nro_dias                    0
geo4_destino_id           108
geo4_nombre               108
comuna_localidad          108
dtype: int64

In [12]:
chx_notna = chx[chx.comuna_localidad.notna()]
chx_isna = chx[chx.comuna_localidad.isna()]
chx_isna.drop(['geo4_destino_id',	'geo4_nombre',	'comuna_localidad'], axis = 1, inplace = True)
print(f""" isna = {chx_isna.shape} 
        notna = {chx_notna.shape} 
        total = {chx.shape}  
        
      """)


 isna = (108, 5) 
        notna = (932, 8) 
        total = (1040, 8)  

      


C:\Users\Usuario\AppData\Local\Temp\ipykernel_22172\362553920.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chx_isna.drop(['geo4_destino_id',	'geo4_nombre',	'comuna_localidad'], axis = 1, inplace = True)


In [13]:
chx_isna = chx_isna.merge(rel_localidades, how = 'left', left_on = 'LOCALIDAD DESTINO', right_on = 'geo4_nombre' )
chx = chx_isna.merge(chx_notna, how = 'outer') 
chx_notna = chx[chx.comuna_localidad.notna()]
chx_isna = chx[chx.comuna_localidad.isna()]
chx_isna.drop(['geo4_destino_id',	'geo4_nombre',	'comuna_localidad'], axis = 1, inplace = True)
print(f""" isna = {chx_isna.shape} 
        notna = {chx_notna.shape} 
        total = {chx.shape}  
        
      """)


 isna = (33, 5) 
        notna = (1008, 8) 
        total = (1041, 8)  

      


C:\Users\Usuario\AppData\Local\Temp\ipykernel_22172\2827717489.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chx_isna.drop(['geo4_destino_id',	'geo4_nombre',	'comuna_localidad'], axis = 1, inplace = True)


In [14]:
chx.sample()

,LOCALIDAD DESTINO,valor_externo,servicio_transporte_id,geo4_origen_id,nro_dias,geo4_destino_id,geo4_nombre,comuna_localidad
192,CONSTITUCION - LA PITRA,CONSTITUCION - LA PITRA,8,333,7,754.0,LA PITRA,CONSTITUCION - LA PITRA


In [476]:
len('Obtenido desde Excel con promesas de entregas de Chile Express (Facilitado por ejecutiva a Fernando)')

100

In [16]:
exp = chx[['valor_externo', 'geo4_destino_id']].drop_duplicates()
exp.columns = ['valor_externo', 'geografia_id']
exp['origen'] = 'CHX-EPE' 
exp['observaciones'] = 'Informacion obtenida desde Excel con promesas de entregas de Chile Express (Facilitado por ejecutiva a Fernando en 2026-04)'
exp['nivel_interno'] = 4

exp.shape
exp.sample()
exp.isna().sum()

# exp.to_sql('relacion_geografia', get_engine('dwh'), index = False, if_exists= 'append', schema = 'dwh' )

valor_externo     0
geografia_id     25
origen            0
observaciones     0
nivel_interno     0
dtype: int64

In [20]:
# exp['LOCALIDAD DESTINO'].value_counts()

In [21]:
# chx_isna['LOCALIDAD DESTINO'].unique()

In [409]:
chx.sample()

,LOCALIDAD DESTINO,servicio_transporte_id,geo4_origen_id,nro_dias,geo3_nombre,localidad
212,ANCUD,9,333,3,ANCUD,ANCUD


In [388]:
# chx['geo4_destino_id'] = chx.apply(lambda x: d[x['localidad']] if x['localidad'] in d.keys() else x['geo4_destino_id'] ,axis  =1 )

In [391]:
chx.shape

(1040, 12)

In [392]:
# rel_localidades[rel_localidades.geo3_nombre.str.contains('PALMI')]

In [22]:
chx.sample()

,LOCALIDAD DESTINO,valor_externo,servicio_transporte_id,geo4_origen_id,nro_dias,geo4_destino_id,geo4_nombre,comuna_localidad
542,MOLINA - ITAHUE,MOLINA - ITAHUE,8,333,7,493.0,ITAHUE,MOLINA - ITAHUE


In [23]:
# d = {'COYHAIQUE' : 55, 
#      'COLLO' : 703, 
#      'POCONCHI' : 704,
#      'SOMO ALTO' : 707, 
#      'SOMO BAJO' : 708, 
#      'LA PATONERA' : 727, 
#      'PAIHUANO' : 197, 
#      'MONTE GRANDE' : 405,
#      'LA PALMA' : 435,
#      'SAN PEDRO QUINTA REGION' : 391,
#      'LLAY-LLAY' : 143, 
#      'LA CALERA' : 23, 
#      'PLACILLA QUINTA REGION' : 409

#     }

# chx['geo4_destino_id'] = chx.apply(lambda x: d[x['localidad']] if x['localidad'] in d.keys() else x['geo4_destino_id'], axis = 1 )
# chx.drop(['geo3_codigo', 'geo3_nombre', 'geo4_nombre', 'LOCALIDAD DESTINO', 'localidad'], axis = 1, inplace = True)

In [311]:
chx.sample()

,servicio_transporte_id,geo4_origen_id,nro_dias,geo4_destino_id
172,5,333,1,69.0


In [326]:
chx[(chx.servicio_transporte_id == 5 ) & (chx.geo4_destino_id == 478)]

,servicio_transporte_id,geo4_origen_id,nro_dias,geo4_destino_id
135,5,333,1,478.0
1020,5,333,1,478.0


In [346]:
# rel_localidades[rel_localidades.comuna_localidad.str.contains('MELIPILLA')]

In [345]:
# chx2[chx2['LOCALIDAD DESTINO'].str.lower().str.contains('melipilla')].sort_values('LOCALIDAD DESTINO')

In [267]:
aux = chx[chx.geo4_destino_id.isna()][['LOCALIDAD DESTINO', 'localidad', 'geo3_nombre']].drop_duplicates()
aux.shape

(13, 3)

In [282]:
rel_localidades.sample()

,geo4_destino_id,geo3_codigo,geo4_nombre
259,261,06101,RANCAGUA


In [320]:
chx.value_counts()

servicio_transporte_id  geo4_origen_id  nro_dias  geo4_destino_id
10                      333             10        581.0              4
                                                  633.0              4
                                                  662.0              4
                                                  692.0              4
5                       333             1         478.0              2
6                       333             1         36.0               2
                                                  560.0              2
                                                  478.0              2
                                                  481.0              2
8                       333             7         606.0              2
                                                  648.0              2
10                      333             10        525.0              2
                                                  545.0              2
           

In [269]:
aux

,LOCALIDAD DESTINO,localidad,geo3_nombre
23,COYHAIQUE,COYHAIQUE,COYHAIQUE
27,SAN PEDRO DE ATACAMA - COLLO,COLLO,SAN PEDRO DE ATACAMA
28,SAN PEDRO DE ATACAMA - POCONCHI,POCONCHI,SAN PEDRO DE ATACAMA
188,RIO HURTADO - SOMO ALTO,SOMO ALTO,RIO HURTADO
189,RIO HURTADO - SOMO BAJO,SOMO BAJO,RIO HURTADO
343,PICHIDEGUA - LA PATONERA,LA PATONERA,PICHIDEGUA
616,PAIHUANO,PAIHUANO,PAIHUANO
711,PAIHUANO - MONTE GRANDE,MONTE GRANDE,PAIHUANO
884,LA PALMA,LA PALMA,LA PALMA
892,SAN PEDRO QUINTA REGION,SAN PEDRO QUINTA REGION,SAN PEDRO QUINTA REGION


In [24]:
rc = get_consulta('dwh', """ 
select 
    g4.id as rc_nivel4_id, 
    g4.nivel3_id, 
    g4.nombre as geo4_nombre--,
    --split_part(g4.codigo, ' - ', 1)  as codigo, 
    --count(g4.*) over (partition by g4.nivel3_id) as ncod 
from 
    dwh.dim_geografia_nivel4 g4


""") #rel_comunas.copy()
rc['geo4_nombre'] = rc.geo4_nombre.apply(lambda x: unidecode(x.upper()).strip() ) 
rc.sample(4)

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,rc_nivel4_id,nivel3_id,geo4_nombre
569,586,232,QUETROLEUFU
443,450,140,SAN FRANCISCO DE LIMACHE
234,236,236,PUERTO OCTAY
362,395,30,TROVOLHUE


In [197]:
aux = aux.merge(rc, how = 'left', left_on = 'geo3_nombre', right_on = 'geo4_nombre')

In [198]:
aux.shape

(86, 6)

In [199]:
aux.isna().sum()

LOCALIDAD DESTINO     0
localidad             0
geo3_nombre           0
rc_nivel4_id         12
nivel3_id            12
geo4_nombre          12
dtype: int64

In [200]:
d3 = {
       'PUERTO AYSEN' : 14 , 
       'PUERTO CISNES' : 49, 
       'COYHAIQUE' : 55,
       'MARIA ELENA SOQUIMICH' : 167, 
       'OHIGGINS' : 194, 
       'TIL TIL' : 320 ,
       'ANDACOLLO HOLDING' : 7, 
       'PAIHUANO' : 197, 
       'LA PALMA' : 435 ,
       'SAN PEDRO QUINTA REGION' : 254, 
       'LLAY-LLAY' : 143, 
       'LA CALERA' : 23,
       'PLACILLA QUINTA REGION' : 333, 
       'CALETA TORTEL' : 327,
       'MONTE GRANDE' : 405, 
       'CASUTO' : 453
}
aux['nivel3_id'] = aux.apply(lambda x: d3[x['localidad']] if x['localidad'] in d3.keys() else x['nivel3_id'] , axis = 1)

In [201]:
aux = aux.drop_duplicates() 
aux.sample(3)
# aux.shape

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre
39,LOS ANGELES - LAS QUINTAS,LAS QUINTAS,LOS ANGELES,154.0,154.0,LOS ANGELES
69,PUCHUNCAVI - CALETA HORCON,CALETA HORCON,PUCHUNCAVI,231.0,231.0,PUCHUNCAVI
63,CURACAVI - LOLENCO,LOLENCO,CURACAVI,75.0,75.0,CURACAVI


In [202]:
aux = aux[aux['LOCALIDAD DESTINO'].str.contains('-')]

In [203]:
nivel3_id = 346
sql_cod = """ select distinct 
split_part(codigo, ' - ', 1) || ' - 00' || CAST(count(*) OVER (PARTITION BY nivel3_id) AS text) as codigo
     from dwh.dim_geografia_nivel4 where nivel3_id = %s """
cod = get_consulta('dwh', sql_cod, params = (nivel3_id,)).codigo.iloc[0]
cod

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:65: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion, params = params)


'05405 - 003'

In [204]:
aux.sample()
aux.head(1)

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre
3,PAIHUANO - MONTE GRANDE,MONTE GRANDE,PAIHUANO,NaN,405.0,NaN


In [743]:
# aux[aux.localidad == 'EL TAMBO']

In [205]:
aux['fecha_actualizacion_origen'] = datetime.now(timezone.utc)
aux.shape

(70, 7)

In [206]:
aux.sample()

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre,fecha_actualizacion_origen
49,LOS ANGELES - SANTA FE,SANTA FE,LOS ANGELES,154.0,154.0,LOS ANGELES,2026-04-21 13:15:19.958974+00:00


In [207]:
aux.iloc[58:62]

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre,fecha_actualizacion_origen
63,CURACAVI - LOLENCO,LOLENCO,CURACAVI,75.0,75.0,CURACAVI,2026-04-21 13:15:19.958974+00:00
64,CURACAVI - LOS PANGUILES,LOS PANGUILES,CURACAVI,75.0,75.0,CURACAVI,2026-04-21 13:15:19.958974+00:00
65,PUCHUNCAVI - PUCALAN,PUCALAN,PUCHUNCAVI,231.0,231.0,PUCHUNCAVI,2026-04-21 13:15:19.958974+00:00
66,ALGARROBO - MIRASOL,MIRASOL,ALGARROBO,1.0,1.0,ALGARROBO,2026-04-21 13:15:19.958974+00:00


In [208]:
aux[aux.rc_nivel4_id.isna()]

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre,fecha_actualizacion_origen
3,PAIHUANO - MONTE GRANDE,MONTE GRANDE,PAIHUANO,NaN,405.0,NaN,2026-04-21 13:15:19.958974+00:00
54,LLAY-LLAY,LLAY-LLAY,LLAY-LLAY,NaN,143.0,NaN,2026-04-21 13:15:19.958974+00:00


In [209]:
aux = aux[aux.rc_nivel4_id.notna()]

In [211]:
aux.shape

(68, 7)

In [210]:
aux.sample()

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre,fecha_actualizacion_origen
34,LINARES - SANTA ANA DE GUAPI,SANTA ANA DE GUAPI,LINARES,141.0,141.0,LINARES,2026-04-21 13:15:19.958974+00:00


In [104]:
from config.conexiones import ejecutar_query

In [213]:
# # dejando lat & lon null 
# for index, row in aux.iterrows():
#     try: 
#         nombre = row['localidad'].title() 
#         nombre3 = row['geo3_nombre'].title()
#         nivel3_id = row['nivel3_id']
#         fao = row['fecha_actualizacion_origen']
#         sql_cod = """ select distinct 
#     split_part(codigo, ' - ', 1) || ' - 00' || CAST(count(*) OVER (PARTITION BY nivel3_id) AS text) as codigo
#         from dwh.dim_geografia_nivel4 where nivel3_id = %s """
#         cod = get_consulta('dwh', sql_cod, params = (nivel3_id,)).codigo.iloc[0]
#         # lat, lon = get_ubi(nombre, nombre3)
#         # print(cod, lat, lon, nombre, nivel3_id)
#         sql_insert = """ INSERT INTO  dwh.dim_geografia_nivel4 
#                         (pais_id, codigo, nombre, nombre_tableau, nombre_nivel,  nivel3_id, fecha_actualizacion_origen )
#                         values (1, %s, %s, %s, 'Localidad', %s , %s)
#         """
#         ejecutar_query('dwh', sql_insert, params = (cod, nombre, nombre, nivel3_id, fao))
#         print(nombre, nombre3, nivel3_id, fao)
#     except Exception as e: 
#         print(nombre, nombre3, nivel3_id, fao)
#         print(str(e))
#         pass

## Buscamos Lat y Lon NULL

In [256]:
refill = get_consulta('dwh', """ 
                      select 
                            g4.id, 
                            g4.nombre, 
                            g3.nombre as nombre3
                      from
                            dwh.dim_geografia_nivel4 g4 left join 
                            dwh.dim_geografia_nivel3 g3 on g3.id = g4.nivel3_id
                      where 
                            g4.lat is null or
                            g4.lon is null 
                      """ )

refill.isna().sum()

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


id         0
nombre     0
nombre3    0
dtype: int64

In [228]:
refill.shape
# import time

(164, 3)

In [234]:
refill.head()

,id,nombre,nombre3
0,702,Alto Talor,San Pedro de Atacama
1,703,Collo,San Pedro de Atacama
2,704,Poconchi,San Pedro de Atacama
3,705,Maria Elena Soquimich,María Elena
4,707,Somo Alto,Río Hurtado


In [238]:
get_ubi('poconchi', 'San pedro de Atacama')

No se encontraron resultados.


(None, None)

In [237]:
get_ubicacion('poconchi, San pedro de Atacama, Chile')

No se encontraron resultados.


(0, 'Sin resultados', None, None)

In [253]:
id = 711, 
direccion = "olivar bajo, rancagua, Chile"

# e, m, lat, lon = get_ubicacion(direccion) 

lat = -42.405217
lon = -73.778929

sql_update = """ UPDATE 
                dwh.dim_geografia_nivel4
            SET 
                lat = %s, 
                lon = %s 
            where 
                ID = %s    
            """
ejecutar_query('dwh', sql_update, params=(lat,lon,id))

In [254]:
# refill.sort_values('id')

In [257]:
for index, row in refill.sort_values('id').head(15).iterrows(): 
    id = row['id'] 
    print(f"\n {id} \n")
    direccion = f"{row['nombre']}, {row['nombre3']}, Chile"
    e, m, lat, lon = get_ubicacion(direccion) 
    sql_update = """ UPDATE 
                        dwh.dim_geografia_nivel4
                    SET 
                        lat = %s, 
                        lon = %s 
                    where 
                        ID = %s    
                 """
    ejecutar_query('dwh', sql_update, params=(lat,lon,id))
    time.sleep(2)


 702 

No se encontraron resultados.

 704 

Coordenadas de Poconche, San Pedro de Atacama, Chile: Latitud=-22.9676578, Longitud=-68.1949413

 705 

No se encontraron resultados.

 707 

Coordenadas de Samo Alto, Río Hurtado, Chile: Latitud=-30.4090831, Longitud=-70.9383091

 708 

No se encontraron resultados.

 712 

No se encontraron resultados.

 713 

No se encontraron resultados.

 714 

No se encontraron resultados.

 715 

No se encontraron resultados.

 716 

No se encontraron resultados.

 717 

No se encontraron resultados.

 718 

No se encontraron resultados.

 719 

No se encontraron resultados.

 720 

No se encontraron resultados.

 721 

No se encontraron resultados.


In [748]:
# for index, row in aux.iterrows():
#     nombre = row['localidad'].title() 
#     nombre3 = row['geo3_nombre'].title()
#     nivel3_id = row['nivel3_id']
#     fao = row['fecha_actualizacion_origen']
#     sql_cod = """ select distinct 
# split_part(codigo, ' - ', 1) || ' - 00' || CAST(count(*) OVER (PARTITION BY nivel3_id) AS text) as codigo
#      from dwh.dim_geografia_nivel4 where nivel3_id = %s """
#     cod = get_consulta('dwh', sql_cod, params = (nivel3_id,)).codigo.iloc[0]
#     lat, lon = get_ubi(nombre, nombre3)
#     print(cod, lat, lon, nombre, nivel3_id)
#     if pd.notna(lat): 
#         sql_insert = """ INSERT INTO  dwh.dim_geografia_nivel4 
#                         (pais_id, codigo, nombre, nombre_tableau, nombre_nivel, lat, lon, nivel3_id, fecha_actualizacion_origen )
#                         values (1, %s, %s, %s, 'Localidad', %s, %s, %s , %s)
#         """
#         ejecutar_query('dwh', sql_insert, params = (cod, nombre, nombre, lat, lon, nivel3_id, fao))
#     pass

In [650]:
aux.sample()

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre
147,MONTE PATRIA - PALERMO,PALERMO,MONTE PATRIA,176.0,176.0,MONTE PATRIA


In [619]:
chx[chx.localidad.str.lower().str.contains('ays')]

,LOCALIDAD DESTINO,servicio_transporte_id,geo4_origen_id,nro_dias,geo3_nombre,localidad,geo4_destino_id,geo3_codigo,geo4_nombre
10,PUERTO AYSEN,7,333,4,PUERTO AYSEN,PUERTO AYSEN,NaN,NaN,NaN
539,PUERTO AYSEN,11,333,5,PUERTO AYSEN,PUERTO AYSEN,NaN,NaN,NaN


In [632]:
# get_ubicacion("casuto, rinconada, chile")

In [638]:

aux['nivel3_id'] = aux.nivel3_id.astype(int)
aux.sample()

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre
247,ILLAPEL - CARCAMO,CARCAMO,ILLAPEL,112.0,112,ILLAPEL


In [633]:
aux[aux['nivel3_id'].isna()].drop_duplicates().localidad.unique()

array([], dtype=object)

In [634]:
aux[aux['nivel3_id'].isna()].drop_duplicates()

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre


In [607]:
aux.sample()

,LOCALIDAD DESTINO,localidad,geo3_nombre,rc_nivel4_id,nivel3_id,geo4_nombre
280,LAS CABRAS - EL MANZANO,EL MANZANO,LAS CABRAS,135.0,135.0,LAS CABRAS


In [102]:
def get_ubi(localidad, comuna): 
    try: 
        e,m, lat, lon = get_ubicacion(f"{localidad}, {comuna}, Chile") 
    except Exception as e: 
        print(e)
        lat = None
        lon = None
    return lat, lon  

# aux[['lat, lon']] = aux.apply(lambda x: get_ubi(x['localidad'], x['geo3_nombre']), axis = 1)

In [555]:
chx.sample()

,LOCALIDAD DESTINO,servicio_transporte_id,geo4_origen_id,nro_dias,geo3_nombre,localidad,geo4_destino_id,geo3_codigo,geo4_nombre
298,RENAICO - TIJERAL,6,333,1,RENAICO,TIJERAL,NaN,NaN,NaN


In [635]:
chx[chx.geo4_destino_id.isna()].sample()

,LOCALIDAD DESTINO,servicio_transporte_id,geo4_origen_id,nro_dias,geo3_nombre,localidad,geo4_destino_id,geo3_codigo,geo4_nombre
480,MELIPILLA - CODIGUA,10,333,10,MELIPILLA,CODIGUA,NaN,NaN,NaN


In [636]:
chx.shape
np.sort(chx[chx.geo4_destino_id.isna()].localidad.unique())
# chx[chx.geo4_destino_id.isna()].localidad.nunique()

array(['ACCESO NORTE LINARES', 'AEROPUERTO A.M. BENITEZ',
       'AGUAS Y ARENAS', 'ALGARROBITO', 'ALMAHUE VIEJO', 'ALTO TALOR',
       'ALTOVALSOL', 'ANDACOLLO HOLDING', 'ANGOSTURA', 'ANTIGUALA',
       'ARBOLEDA GRANDE', 'ARRAYAN', 'AUQUINCO', 'AURORA', 'BETER',
       'BOBADILLA', 'BUSTAMANTE', 'CABANAS ALTO ILLAPEL', 'CABURGUA',
       'CAHUIL', 'CALETA BOY', 'CALETA HORCON', 'CALETA TORTEL',
       'CAMINO A LA ESPERANZA', 'CAMINO A MANATIALES',
       'CAMINO A PIEDRA BLANCA', 'CANAL DEL PAICO', 'CANDELARIA',
       'CARCAMO', 'CAREN', 'CARMEN BAJO', 'CARMEN DE LAS ROSAS',
       'CARRIRINGUE', 'CARRIZAL', 'CARRIZALILLO', 'CASAS DE LA ESPERANZA',
       'CASMA', 'CASUTO', 'CATO', 'CAUNAO', 'CAYUMAPU',
       'CERRILLOS DE RAPEL', 'CERRO BLANCO', 'CHACURRA', 'CHADA',
       'CHAGUARAL', 'CHALINGUITA', 'CHAMIZA', 'CHAMPA', 'CHANARAL ALTO',
       'CHARRUA', 'CHILLEPIN', 'CHOAPA', 'CHOCALAN', 'CHOCOCAMAYO',
       'CHOLQUI', 'CHOMEDAHUE', 'CHOSHUENCO', 'CHUCHINI', 'COCALAN',
       

### Comunas

In [55]:
rel_comunas = get_consulta('dwh', "select id::integer as geo_destino_id,  geo3_nombre from dwh.dim_geografia")
rel_comunas['geo3_nombre'] = rel_comunas.geo3_nombre.apply(lambda x: unidecode(x).upper())
rel_comunas.sample()

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:61: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


,geo_destino_id,geo3_nombre
37,38,CHAITEN


In [90]:
chx = chx2[chx2.geo3_nombre == chx2.localidad]
chx.shape

(553, 6)

In [91]:
chx = chx.merge(rel_comunas, how = 'left', on = 'geo3_nombre')

In [92]:
chx.sample()

,LOCALIDAD DESTINO,servicio_transporte_id,geo_origen_id,nro_dias,geo3_nombre,localidad,geo_destino_id
401,QUILLON,6,333,1,QUILLON,QUILLON,253.0


In [93]:
chx.shape

(553, 7)

In [96]:
d = { 'ANDACOLLO HOLDING' : 7 ,  
      'COYHAIQUE' : 55,
     'LA CALERA' : 23,  
     'LLAY-LLAY' : 143 , 
     'MARIA ELENA SOQUIMICH' : 167, 
     'PAIHUANO' : 197,
     'PUERTO AYSEN' : 14,  
     'PUERTO CISNES' : 49, 
     'TIL TIL' : 320 }

chx['geo_destino_id'] = chx.apply(lambda x: d[x['LOCALIDAD DESTINO']] if x['LOCALIDAD DESTINO'] in d.keys() else x['geo_destino_id'] , axis = 1)

In [122]:
import numpy as np
# np.sort(chx[chx.geo_destino_id.isna()].geo3_nombre.unique())

In [149]:
cch.localidad.unique()

array(['CHACAO', 'CARAMPANGUE', 'LARAQUETE', 'PUERTO AGUIRRE',
       'PUERTO CHACABUCO', 'ALTO JAHUEL', 'EL RECURSO',
       'VALDIVIA DE PAINE', 'VILUCO', 'PUERTO WILLIAMS', 'MONTE AGUILA',
       'TROVOLHUE', 'SAN SEBASTIAN', 'LA JUNTA', 'CASAS DE CHACABUCO',
       'ESTACION PAIPOTE', 'PAIPOTE', 'TONGOY', 'LOS LAURELES',
       'EL SALVADOR', 'LO MIRANDA', 'EL PAICO', 'LAS CRUCES', 'QUEPE',
       'HUALPIN', 'MELINKA', 'HORNOPIREN', 'LONQUEN', 'SANTA FE DE LAJA',
       'BATUCO', 'MALALHUE', 'QUILIMARI', 'CAPITAN PASTEN', 'PELEQUEN',
       'BOLLENAR', 'MALLARAUCO', 'SAN JOSE DE MELIPILLA', 'EL PALQUI',
       'SAN GREGORIO (XVI)', 'MALLOCO', 'MONTEGRANDE', 'PISCO ELQUI',
       'HOSPITAL', 'CURANIPE', 'MAITENCILLO', 'ENTRE LAGOS',
       'SAN PEDRO DE QUILLOTA', 'EL BELLOTO', 'ACHAO', 'NIPAS', 'ROSARIO',
       'CUMPEO', 'PUERTO RIO TRANQUILO', 'PUERTO INGENIERO IBANEZ',
       'LLO LLEO', 'NOS', 'ISLA NEGRA', 'LABRANZA', 'BARROS ARANA',
       'HUERTO FAMILIARES', 'MONTENEGRO', '

In [110]:
e = np.sort(list(set(cch.localidad.unique().tolist() + chx[chx.geo_destino_id.isna()].geo3_nombre.unique().tolist())))

In [112]:
len(e)
e

array(['ACHAO', 'ALERCE', 'ALTO JAHUEL', 'ARTIFICIO', 'BALMACEDA',
       'BARRANCAS', 'BARROS ARANA', 'BATUCO', 'BOLLENAR',
       'BRISAS DE SANTO DOMINGO', 'CACHAGUA', 'CAJON', 'CAPITAN PASTEN',
       'CARAMPANGUE', 'CASAS DE CHACABUCO', 'CATAPILCO', 'CHACAO',
       'CHICUREO', 'CHOLGUAN', 'CUMPEO', 'CURANIPE', 'DICHATO',
       'EL BELLOTO', 'EL MELON', 'EL PAICO', 'EL PALQUI', 'EL RECURSO',
       'EL SALVADOR', 'EL TABITO', 'ENTRE LAGOS', 'ESTACION DOMEIKO',
       'ESTACION PAIPOTE', 'FRUTILLAR ORIENTE', 'FRUTILLAR SUR',
       'HORNOPIREN', 'HOSPITAL', 'HUALPIN', 'HUEPIL', 'HUERTO FAMILIARES',
       'ISLA NEGRA', 'ISLA TEJA', 'LA JUNTA', 'LA PALMA', 'LA PAZ',
       'LABRANZA', 'LARAQUETE', 'LAS CANCHAS', 'LAS CRUCES', 'LAS TACAS',
       'LICANRAY', 'LIRQUEN', 'LLO LLEO', 'LO MIRANDA', 'LONGOVILO',
       'LONQUEN', 'LONTUE', 'LOS LAURELES', 'MAITENCILLO', 'MALALHUE',
       'MALLARAUCO', 'MALLOCO', 'MELINKA', 'MININCO', 'MONTE AGUILA',
       'MONTEGRANDE', 'MONTENEGRO', '

In [151]:
cch.sample()

,geo_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia
186,341,RANCAGUA,MALLOA,PELEQUEN,LOCALIDAD,Sin Distribución,1


In [154]:
cch[cch['localidad'].str.upper() == l]

,geo_origen_id,planta,comuna,localidad,tipo,Domiciliario,Sucursal/Agencia
285,341,CASTRO,QUINCHAO,ACHAO,LOCALIDAD,2,2


In [153]:
l = 'ACHAO'
chx2[chx2.localidad == l]
# chx[chx.localidad == l]

,LOCALIDAD DESTINO,servicio_transporte_id,geo_origen_id,nro_dias,geo3_nombre,localidad
262,ACHAO,9,333,3,ACHAO,ACHAO


In [157]:
exp = pd.DataFrame({'nombre' : e})
exp = exp.merge(cch[['localidad','comuna']], how = 'left', left_on = 'nombre',  right_on = 'localidad').sort_values('nombre')

In [160]:
exp.comuna.isna().value_counts()

comuna
False    75
True     31
Name: count, dtype: int64

In [176]:
exp.head(5)

,nombre,localidad,comuna
0,ACHAO,ACHAO,QUINCHAO
1,ALERCE,NaN,NaN
2,ALTO JAHUEL,ALTO JAHUEL,BUIN
3,ARTIFICIO,NaN,NaN
4,BALMACEDA,NaN,NaN


In [184]:
llat = []
llon = []
for index, row in exp.iterrows(): 
    nombre = row['nombre']
    comuna = row['comuna'] 
    if pd.isna(comuna): 
        lat = None; lon = None; 
        llat.append(lat)
        llon.append(lon)
        continue
    direccion = f"{nombre}, {comuna}, Chile"
    direccion = direccion.title()
    print(direccion)
    e,m, lat, lon = get_ubicacion(direccion)
    print(lat, lon)
    llat.append(lat)
    llon.append(lon)


Achao, Quinchao, Chile
Coordenadas de Achao, Quinchao, Chile: Latitud=-42.4710010, Longitud=-73.4880780
-42.4710010 -73.4880780
Alto Jahuel, Buin, Chile
Coordenadas de Alto Jahuel, Buin, Chile: Latitud=-33.7320044, Longitud=-70.6843181
-33.7320044 -70.6843181
Barros Arana, Teodoro Schmidt, Chile
Coordenadas de Barros Arana, Teodoro Schmidt, Chile: Latitud=-38.9858471, Longitud=-72.9213764
-38.9858471 -72.9213764
Batuco, Lampa, Chile
Coordenadas de Batuco, Lampa, Chile: Latitud=-33.2398523, Longitud=-70.8143733
-33.2398523 -70.8143733
Bollenar, Melipilla, Chile
Coordenadas de Bollenar, Melipilla, Chile: Latitud=-33.5696545, Longitud=-71.2119606
-33.5696545 -71.2119606
Cajon, Vilcun, Chile
Coordenadas de Cajon, Vilcun, Chile: Latitud=-38.6775553, Longitud=-72.5031812
-38.6775553 -72.5031812
Capitan Pasten, Lumaco, Chile
No se encontraron resultados.
None None
Carampangue, Arauco, Chile
Coordenadas de Carampangue, Arauco, Chile: Latitud=-37.2574546, Longitud=-73.2420686
-37.2574546 -73.24

In [188]:
len(llat)
exp.shape
# llat

(106, 3)

In [189]:
    # break
exp['lat'] = llat
exp['lon'] = llon

In [193]:
rel_comunas = get_consulta('dwh', "select id::integer as geo_nivel3_id, upper(nombre) as comuna from dwh.dim_geografia_nivel3")
rel_comunas.sample()

,geo_nivel3_id,comuna
103,104,HIJUELAS


In [198]:
exp = exp.merge(rel_comunas, how = 'left', on = 'comuna')

In [206]:
exp['geo_nivel3_id'] = exp.geo_nivel3_id.fillna(0).astype(int).replace(0, None)
exp.sample()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_1884\1574556982.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exp['geo_nivel3_id'] = exp.geo_nivel3_id.fillna(0).astype(int).replace(0, None)


,nombre,localidad,comuna,lat,lon,geo_nivel3_id
38,HUERTO FAMILIARES,HUERTO FAMILIARES,TILTIL,-33.1352898,-70.7996764,320


In [207]:
exp1 = exp[(exp.lat.notna()) & (exp.geo_nivel3_id.notna())] 
exp.shape

(106, 6)

In [208]:
exp1.shape

(51, 6)

In [229]:
exp1.sample()

,nombre,lat,lon,geo_nivel3_id,nombre_tableau,nombre_nivel,pais_id,fecha_actualizacion_origen
26,El Recurso,-33.6909354,-70.7030696,15,El Recurso,Localidad,1,2026-04-13 21:23:39.365837+00:00


In [228]:
exp1['fecha_actualizacion_origen'] = datetime.now(timezone.utc)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_1884\1142960587.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp1['fecha_actualizacion_origen'] = datetime.now(timezone.utc)


In [231]:
exp1['nombre'] = exp.nombre.str.title()
exp1['nombre_tableau'] = exp1.nombre
exp1['nombre_nivel'] = 'Localidad'
exp1['pais_id'] = 1
exp1.rename(columns = {'geo_nivel3_id': 'nivel3_id'}, inplace = True)

# exp1.drop(['localidad', 'comuna'], axis = 1, inplace = True)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_1884\3501146102.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp1['nombre'] = exp.nombre.str.title()
C:\Users\Usuario\AppData\Local\Temp\ipykernel_1884\3501146102.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp1['nombre_tableau'] = exp1.nombre
C:\Users\Usuario\AppData\Local\Temp\ipykernel_1884\3501146102.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value

In [232]:
exp1.to_sql('dim_geografia_nivel4', get_engine('dwh'), if_exists = 'append', index = False, schema = 'dwh')

51

In [220]:
exp2 = exp[~exp.nombre.str.upper().isin(exp1.nombre.str.upper())]
exp2.shape

(55, 6)

In [234]:
exp3 = exp2[exp2.lat.notna()]
exp3.shape

(18, 6)

In [235]:
exp3

,nombre,localidad,comuna,lat,lon,geo_nivel3_id
11,CAJON,CAJON,VILCUN,-38.6775553,-72.5031812,None
19,CUMPEO,CUMPEO,RIO CLARO,-35.2816338,-71.2586749,None
21,DICHATO,DICHATO,TOME,-36.5486269,-72.9364378,None
22,EL BELLOTO,EL BELLOTO,QUILPUE,-33.0431718,-71.4065260,None
31,ESTACION PAIPOTE,ESTACION PAIPOTE,COPIAPO,-27.4089235,-70.2719343,None
34,HORNOPIREN,HORNOPIREN,HUALAIHUE,-41.9660516,-72.4706795,None
57,MAITENCILLO,MAITENCILLO,PUCHUNCAVI,-32.6441845,-71.4329357,None
64,MONTEGRANDE,MONTEGRANDE,PAIHUANO,-30.0938591,-70.4944432,None
66,NIPAS,NIPAS,RANQUIL,-36.6050640,-72.5343744,None
70,PAIPOTE,PAIPOTE,COPIAPO,-27.4147265,-70.2749806,None


In [508]:
e,m, lat, lon = get_ubicacion("estacion paipote, copiapo, chile ")

Coordenadas de estacion paipote, copiapo, chile : Latitud=-27.4089235, Longitud=-70.2719343


In [506]:
e,m, lat, lon = get_ubicacion("paipote, copiapo, chile ")

Coordenadas de paipote, copiapo, chile : Latitud=-27.4147265, Longitud=-70.2749806


In [509]:

id = 452
print( lat,lon, id)
sql_update = """ 

UPDATE 
    dwh.dim_geografia_nivel4 g4 
SET 
    lat = %s, 
    lon = %s 
where
    id = %s

 """

ejecutar_query('dwh',sql_update, params = ( lat,lon, id))

-27.4089235 -70.2719343 452


In [295]:
from config.conexiones import ejecutar_query

In [325]:
z = get_consulta('sap', """ SELECT * FROM "PRD_INTERANDINA".OITM ORDER BY "CreateDate" Desc limit 1 """ )
[col for col in z.columns if 'size' in col.lower()]

C:\Users\Usuario\Desktop\Proyectos\Logistica\config\conexiones.py:61: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conexion)


['U_GSP_Size', 'U_GSP_GroupSize', 'U_GSP_SUBSIZE']

In [312]:
exp4 = exp2[exp2.lat.isna()]
exp4['direccion'] = exp.nombre.str.title() + ', ' +  exp.comuna.str.title()
exp4.sort_values('geo_nivel3_id').tail(5)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_1884\2046518718.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp4['direccion'] = exp.nombre.str.title() + ', ' +  exp.comuna.str.title()


,nombre,localidad,comuna,lat,lon,geo_nivel3_id,direccion
80,PUEBLO SECO,NaN,NaN,None,None,None,NaN
89,QUIRIQUINA,NaN,NaN,None,None,None,NaN
94,SAN FRANCISCO DE LIMACHE,NaN,NaN,None,None,None,NaN
95,SAN FRANCISCO DE MOSTAZAL,NaN,NaN,None,None,None,NaN
99,SAN PEDRO QUINTA REGION,NaN,NaN,None,None,None,NaN


In [238]:
exp4.shape

(37, 6)

In [100]:
import requests

def get_ubicacion(direccion): 


    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": direccion,
        "format": "json",
        "limit": 1
    }

    # IMPORTANTE: añadir un User-Agent
    headers = {
        "User-Agent": "MiAppGeocodificacion/1.0 (tu_email@ejemplo.com)"
    }

    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
        data = response.json()
        if data:
            lat = data[0]["lat"]
            lon = data[0]["lon"]
            print(f"Coordenadas de {direccion}: Latitud={lat}, Longitud={lon}")
            return 1,"OK", lat, lon
        else:
            print("No se encontraron resultados.")
            return 0, "Sin resultados", None, None
    else:
        print("Error en la petición:", response.status_code)
        return -1, "", None, None

In [75]:
aux.sample()
aux[aux['LOCALIDAD DESTINO'].str.lower().str.contains('limache')]

,LOCALIDAD ORIGEN,LOCALIDAD DESTINO,PRODUCTO,TIEMPO ENTREGA,DIAS,Unnamed: 5,SERVICIO,DIAS.1
992,VALPARAISO,SAN FRANCISCO DE LIMACHE,ENCOMIENDA,EXPRESS,1,NaN,NaN,NaN
1014,VALPARAISO,LIMACHE,ENCOMIENDA,EXPRESS,1,NaN,NaN,NaN
